In [ ]:
import os
import re
import glob
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import numpy as np
import geopandas as gpd
from shapely.geometry import shape, Polygon, MultiPolygon
import rasterio
from rasterio.mask import mask as rio_mask
from rasterio.windows import from_bounds
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from typing import Optional

In [ ]:
# ------------------------- USER CONFIG -------------------------
RASTER_DIR = r"/S2_images/combined_3"          # contains S2_n_yyyymmdd.tif files
SHAPE_DIR  = r"/tr_shapes"       # contains Fields_with_changeFlag_samplen.shp files
OUT_DIR    = r"/tr_data_cutted"   # where im_* outputs go
MIN_VALID_PCT = 10.0                      # threshold (%) for "valid" previous images
DATE_FIELD_REGEX = r"^\d{4}-\d{2}-\d{2}$" # date-like fields to consider
ID_FIELD = "id"                           # polygon id field name
AREA_FIELD = "area"                     # polygon area field (meters)
# ---------------------------------------------------------------

In [ ]:
date_field_pattern = re.compile(DATE_FIELD_REGEX)

def parse_sample_id_from_shp(shp_path: str) -> int:
    """
    Extracts sample id n from filename pattern: Fields_with_changeFlag_samplen.shp
    """
    m = re.search(r"Fields_with_changeFlag_sample(\d+)\.shp$", os.path.basename(shp_path))
    if not m:
        raise ValueError(f"Cannot parse sample id from shapefile name: {shp_path}")
    return int(m.group(1))

def build_raster_index_for_sample(sample_id: int) -> dict:
    """
    For given sample n, index rasters by date string 'YYYY-MM-DD' -> path
    Files are S2_n_yyyymmdd.tif
    """
    patt = os.path.join(RASTER_DIR, f"S2_{sample_id}_*.tif")
    mapping = {}
    for p in glob.glob(patt):
        m = re.search(r"S2_(\d+)_(\d{8})\.tif$", os.path.basename(p))
        if not m:
            continue
        n = int(m.group(1))
        if n != sample_id:
            continue
        yyyymmdd = m.group(2)
        dt = datetime.strptime(yyyymmdd, "%Y%m%d").date()
        mapping[dt.isoformat()] = p
    return mapping

def get_date_fields(gdf: gpd.GeoDataFrame) -> list:
    """
    Return sorted list of columns that look like YYYY-MM-DD.
    """
    date_cols = [c for c in gdf.columns if date_field_pattern.match(c)]
    # sort by chronological order
    date_cols_sorted = sorted(date_cols, key=lambda x: datetime.strptime(x, "%Y-%m-%d"))
    return date_cols_sorted

def percent_valid_pixels(poly_geom, raster_path: str) -> Optional[float]:
    """
    Compute percentage of valid pixels inside poly for the given raster.
    'Valid' = Band1 finite and != 0. Handles NaNs.
    Returns None if raster missing or empty overlap; else percent (0..100).
    """
    if not os.path.exists(raster_path):
        return None
    try:
        with rasterio.open(raster_path) as src:
            # Crop by polygon; return masked array with True mask outside
            arr, out_transform = rio_mask(src, [poly_geom.__geo_interface__], crop=True, filled=False)
            if arr.size == 0:
                return 0.0
            band1 = arr[0]  # masked array
            # inside-polygon pixels are those not masked
            inside = ~band1.mask
            total_inside = int(inside.sum())
            if total_inside == 0:
                return 0.0
            # valid if finite and != 0
            # Need raw values; masked entries may contain junk; use inside mask
            b1_values = band1.data[inside]
            valid = np.isfinite(b1_values) & (b1_values != 0)
            pct = 100.0 * (valid.sum() / total_inside)
            return float(pct)
    except Exception as e:
        # Could log e
        return None

def read_masked_on_ref_grid(poly_geom, ref_path: str):
    """
    Read and crop an image to polygon on its own grid, returning array (bands,H,W),
    transform, crs and a boolean mask (True inside polygon on the returned grid).
    Outside-of-polygon pixels in array will be left as-is; we will zero them after.
    We also return a polygon mask rasterized on this grid so we can enforce zeros.
    """
    with rasterio.open(ref_path) as src:
        # mask returns masked array with same dtype; we keep masked to infer inside pixels
        arr, out_transform = rio_mask(src, [poly_geom.__geo_interface__], crop=True, filled=False)
        crs = src.crs
        # Build a strict inside-polygon boolean mask on this cropped grid (no nodata logic)
        H, W = arr.shape[1], arr.shape[2]
        # Mask from the first band’s mask (True means outside)
        inside = ~arr[0].mask
        return arr, out_transform, crs, inside

def reproject_to_ref_grid(src_path: str, dst_crs, dst_transform, width, height, resampling=Resampling.nearest):
    """
    Read a source image into the provided reference grid (CRS, transform, size).
    Returns ndarray (bands,H,W). Does not apply polygon mask here.
    """
    with rasterio.open(src_path) as base:
        # Ensure 5 bands exist
        if base.count < 5:
            raise ValueError(f"{src_path} has {base.count} bands; expected at least 5.")
        with WarpedVRT(
            base,
            crs=dst_crs,
            transform=dst_transform,
            height=height,
            width=width,
            resampling=resampling,
        ) as vrt:
            data = vrt.read(indexes=list(range(1, 6)))
            return data

def write_10band_tif(out_path: str, ref_meta: dict, data10: np.ndarray):
    """
    Write 10-band GeoTIFF using reference metadata (CRS, transform, dtype).
    data10 shape = (10, H, W).
    """
    meta = ref_meta.copy()
    meta.update(
        {
            "count": 10,
            "dtype": str(data10.dtype),
            "compress": "deflate",
            "tiled": True,
            "interleave": "band",
            "BIGTIFF": "IF_SAFER",
        }
    )
    with rasterio.open(out_path, "w", **meta) as dst:
        dst.write(data10)

def to_yyyymmdd(date_iso: str) -> str:
    return date_iso.replace("-", "")

def main():
    shp_files = sorted(glob.glob(os.path.join(SHAPE_DIR, "Fields_with_changeFlag_sample*.shp")))
    if not shp_files:
        raise SystemExit("No shapefiles found with pattern Fields_with_changeFlag_samplen.shp")

    for shp_path in shp_files:
        sample_id = parse_sample_id_from_shp(shp_path)
        print(f"\n=== Processing sample {sample_id} ===")
        raster_index = build_raster_index_for_sample(sample_id)
        if not raster_index:
            print(f"  No rasters found for sample {sample_id}; skipping.")
            continue

        gdf = gpd.read_file(shp_path)
        if ID_FIELD not in gdf.columns:
            raise ValueError(f"{ID_FIELD} field not found in {shp_path}")
        if AREA_FIELD not in gdf.columns:
            raise ValueError(f"{AREA_FIELD} field not found in {shp_path}")

        date_fields = get_date_fields(gdf)
        if len(date_fields) < 2:
            print(f"  Found fewer than 2 date fields in {shp_path}; skipping.")
            continue

        # Precompute percent valid per polygon per date
        print("  Computing valid-pixel percentages per polygon/date...")
        valid_pct = defaultdict(dict)  # valid_pct[poly_idx][date_iso] = pct or None

        for i, row in gdf.iterrows():
            geom = row.geometry
            # Ensure polygon or multipolygon
            if geom is None or geom.is_empty:
                continue
            # Compute for dates that actually exist in raster_index
            for d in date_fields:
                if d not in raster_index:
                    valid_pct[i][d] = None
                    continue
                pct = percent_valid_pixels(geom, raster_index[d])
                valid_pct[i][d] = pct

        # Now iterate each polygon and each date (from second onward)
        print("  Cutting and writing paired 10-band images...")
        for i, row in gdf.iterrows():
            geom = row.geometry
            if geom is None or geom.is_empty:
                continue

            poly_id = row[ID_FIELD]
            area_m = row[AREA_FIELD]

            # Chronological dates:
            d_sorted = date_fields

            # Build a list of earlier dates that satisfy MIN_VALID_PCT for this polygon
            # We'll check per date-of-interest dynamically (only earlier ones are eligible).
            for idx in range(1, len(d_sorted)):
                d_cur = d_sorted[idx]              # date-of-interest (YYYY-MM-DD)
                if d_cur not in raster_index:
                    continue  # no raster

                cl_value = row[d_cur]
                try:
                    cl_int = int(cl_value)
                except Exception:
                    # If NaN or missing, treat as 0
                    cl_int = 0

                # Determine how many previous images we need
                need_prev = 6 if cl_int == 1 else 1

                # Gather earlier dates with >= MIN_VALID_PCT
                earlier = d_sorted[:idx]
                # Sort earlier by recency descending so we pick the closest previous first
                earlier_sorted = sorted(earlier, key=lambda x: datetime.strptime(x, "%Y-%m-%d"), reverse=True)

                eligible_prev = []
                for dprev in earlier_sorted:
                    pct = valid_pct[i].get(dprev, None)
                    if pct is None:
                        continue
                    if pct >= MIN_VALID_PCT and dprev in raster_index:
                        eligible_prev.append(dprev)
                    if len(eligible_prev) >= need_prev:
                        break

                if len(eligible_prev) == 0:
                    # nothing to pair with; skip this date for this polygon
                    continue

                # Open date-of-interest as reference grid and read masked/cropped
                ref_path = raster_index[d_cur]
                try:
                    with rasterio.open(ref_path) as ref_src:
                        # masked crop on ref grid
                        ref_arr, ref_transform, ref_crs, inside_mask = read_masked_on_ref_grid(geom, ref_path)
                        if ref_arr.size == 0:
                            continue

                        # Determine dtype (keep the same as source)
                        out_dtype = ref_src.dtypes[0]
                        H, W = ref_arr.shape[1], ref_arr.shape[2]

                        # Build a strict polygon mask on the ref grid; 'inside_mask' is True inside polygon
                        # We'll use it to zero out outside pixels:
                        # First, convert masked arrays to raw values; then set outside-of-polygon to 0
                        ref_data = np.array([np.where(inside_mask, b.data, 0) for b in ref_arr], dtype=out_dtype)
                        # If input had NaNs and dtype is float, keep NaNs inside polygon as-is; outside -> 0
                        if np.issubdtype(ref_data.dtype, np.floating):
                            for b_idx in range(ref_data.shape[0]):
                                # Wherever original was masked (outside), set 0 already; keep NaNs inside.
                                pass

                        # For each eligible previous date, build and write a 10-band stack
                        for dprev in eligible_prev:
                            prev_path = raster_index[dprev]

                            # Reproject previous to ref grid
                            prev_data = reproject_to_ref_grid(
                                prev_path,
                                dst_crs=ref_crs,
                                dst_transform=ref_transform,
                                width=W,
                                height=H,
                                resampling=Resampling.nearest,
                            ).astype(out_dtype)

                            # Enforce polygon mask: zero outside for prev_data as well
                            prev_data = np.where(inside_mask, prev_data, 0)

                            # Build 10-band stack: [ref(5), prev(5)]
                            if ref_data.shape[0] < 5 or prev_data.shape[0] < 5:
                                # Safety check; should not happen given assumptions
                                continue
                            stack10 = np.concatenate([ref_data[:5], prev_data[:5]], axis=0)

                            # Compose output name
                            y1 = to_yyyymmdd(d_cur)
                            y2 = to_yyyymmdd(dprev)
                            out_name = f"im_{sample_id}_{cl_int}_{poly_id}_{y1}_{y2}_{int(round(float(area_m)))}.tif"
                            out_path = os.path.join(OUT_DIR, out_name)

                            # Write using ref metadata
                            meta = ref_src.meta.copy()
                            meta.update({
                                "transform": ref_transform,
                                "crs": ref_crs,
                                "width": W,
                                "height": H,
                            })
                            write_10band_tif(out_path, meta, stack10)
                except Exception as e:
                    # Continue gracefully per polygon/date
                    # You can print(e) or log it if desired
                    continue

In [ ]:
if __name__ == "__main__":
    main()